In [41]:
from utils import simulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import os
import numpy as np
import pandas as pd

train_models= True
ijk_wager_calc= False
ijk_butt_calc= False
ijk2_butt_calc= False
boot_calc= False
B_first_level  = 3
seed = 42
X_pred_point = pd.DataFrame({'X_1': [0], 'X_2': [1], 'X_3': [80], 'X_4': [40]})


##  DFZZZZzzffvbaxihhassealleswassoldaskeinbockmehrvfffffffdvnbbbbb

In [ ]:
### Simulations Parameter 
n_sim = 5
n = int(1001/0.7)
B_RF = int(n*0.7 /2)

data_generation_parameter =   { 'shape_weibull': 1,  'p_1': -0.405, 'p_2': -0.4, 'p_3': -0.05, 'p_4': -0.01, 'n': n,
                                'scale_weibull_base':   200       , 
                                'rate_censoring':       0.5    , 
                                'tau': 1, 
                                'data_type':           'weibull'               ,}

params_rf =         {   'n_estimators':B_RF,                        
                        'max_depth':4,
                        'min_samples_split':5,
                        'max_features': 'log2',
                        'random_state':  seed,
                        'weighted_bootstrapping': True, }

### Start Simulation

In [ ]:
with ProcessPoolExecutor() as executor:
    ### Array to store the results
    wb_test_mse = np.zeros(n_sim)
    rf_test_mse = np.zeros(n_sim)
    rf_pred = np.zeros(n_sim)
    wb_pred = np.zeros(n_sim)
    ijk_wager_var = np.zeros(n_sim)
    ijk_butt_var = np.zeros(n_sim)
    ijk2_butt_var = np.zeros(n_sim)
    boot_var = np.zeros(n_sim)
    portion_zero_weights_train = np.zeros(n_sim)
    true_pred = np.zeros(n_sim)

    futures = [
        executor.submit(
            simulation,
            seed=seed+i,
            B_first_level=B_first_level,
            train_models=train_models,
            ijk_wager_calc=ijk_wager_calc,
            ijk_butt_calc=ijk_butt_calc,
            ijk2_butt_calc=ijk2_butt_calc,
            boot_calc=boot_calc,
            data_generation_parameter=data_generation_parameter,
            params_rf=params_rf,
            X_pred_point = X_pred_point,
        )
        for i in range(n_sim)
    ]

    for i, future in enumerate(tqdm(futures, desc="Simulations", unit="simulation")):
        _wb_test_mse, _rf_test_mse, _wb_pred, _rf_pred, _ijk_wager_var, _ijk_butt_var, _ijk2_butt_var, _boot_var, _portion_zero_weights_train, _true_pred   = future.result()

        #Event-Stats Results
        portion_zero_weights_train[i] = _portion_zero_weights_train

        #Evaluation Results
        wb_test_mse[i] = _wb_test_mse
        rf_test_mse[i] = _rf_test_mse

        #Prediction Results
        wb_pred[i] = _wb_pred[0]
        rf_pred[i] = _rf_pred[0]
        true_pred[i] = _true_pred

        # Standard Deviation Estimates
        ijk_wager_var[i] = _ijk_wager_var
        ijk_butt_var[i] = _ijk_butt_var

results = pd.DataFrame({'wb_pred': wb_pred, 'rf_pred': rf_pred, 'true_pred': true_pred, 
                        'wb_test_mse': wb_test_mse, 'rf_test_mse': rf_test_mse, 'ijk_wager_var': ijk_wager_var, 
                        'ijk_butt_var': ijk_butt_var, 'ijk2_butt_var': ijk2_butt_var, 'boot_var': boot_var, 
                        'portion_zero_weights_train': portion_zero_weights_train})

results

Simulations: 100%|██████████| 5/5 [00:02<00:00,  2.16simulation/s]


,wb_pred,rf_pred,true_pred,wb_test_mse,rf_test_mse,ijk_wager_var,ijk_butt_var,ijk2_butt_var,boot_var,portion_zero_weights_train
0,0.546946,0.525427,0.544683,0.197520,0.199392,0.0,0.0,0.0,0.0,0.276723
1,0.527775,0.547863,0.544683,0.216219,0.213966,0.0,0.0,0.0,0.0,0.277722
2,0.558880,0.630088,0.544683,0.229677,0.239853,0.0,0.0,0.0,0.0,0.326673
3,0.546280,0.543949,0.544683,0.236524,0.241826,0.0,0.0,0.0,0.0,0.314685
4,0.514822,0.463607,0.544683,0.234797,0.233414,0.0,0.0,0.0,0.0,0.304695


In [58]:
### Varainte 1  ( nur die var einbeziehen, die grösser 0 sind  )
path = os.path.abspath('')
experiment_name = f'(n_train){int(n*0.7)}__(B_RF){B_RF}__(B_1){B_first_level}__(zero_weights){np.round(np.mean(portion_zero_weights_train),2)}'
if not os.path.exists(path + '/results/'+experiment_name):
    os.makedirs(path + '/results/'+experiment_name)

result_dict_variante1 = { 'wb_pred_mean': results["wb_pred"].mean() ,
                'wb_emp_std': results["wb_pred"].std(ddof=1) ,
                'rf_pred_mean': results["rf_pred"].mean() ,
                'rf_emp_std': results["rf_pred"].std(ddof=1) ,
                'true_pred': results["true_pred"].mean() ,
                'wb_test_mse_mean': results["wb_test_mse"].mean() ,
                'rf_test_mse_mean': results["rf_test_mse"].mean() ,
                'zero_weights_mean': np.mean(portion_zero_weights_train) ,

                'ijk_wager_std_mean':   np.mean(results['ijk_wager_var'].apply(lambda x: np.sqrt(x) if x >= 0 else np.nan)) ,
                'ijk_butt_std_mean':    np.mean(results['ijk_butt_var'].apply(lambda x: np.sqrt(x) if x >= 0 else np.nan)) ,
                'ijk2_butt_std_mean':   np.mean(results['ijk2_butt_var'].apply(lambda x: np.sqrt(x) if x >= 0 else np.nan)) ,
                'boot_std_mean':        np.mean(results['boot_var'].apply(lambda x: np.sqrt(x) if x >= 0 else np.nan)) ,
               }

### Varainte 1  ( alle var 0 setrzen, die kleiner 0 sind  )
result_dict_variante2 = {   'wb_pred_mean': results["wb_pred"].mean() ,
                            'wb_emp_std': results["wb_pred"].std(ddof=1) ,
                            'rf_pred_mean': results["rf_pred"].mean() ,
                            'rf_emp_std': results["rf_pred"].std(ddof=1) ,
                            'true_pred': results["true_pred"].mean() ,
                            'wb_test_mse_mean': results["wb_test_mse"].mean() ,
                            'rf_test_mse_mean': results["rf_test_mse"].mean() ,
                            'zero_weights_mean': np.mean(portion_zero_weights_train) ,

                            'ijk_wager_std_mean':   np.mean(results['ijk_wager_var'].apply(lambda x: np.sqrt(x) if x >= 0 else 0)) ,
                            'ijk_butt_std_mean':    np.mean(results['ijk_butt_var'].apply(lambda x: np.sqrt(x) if x >= 0 else 0)) ,
                            'ijk2_butt_std_mean':   np.mean(results['ijk2_butt_var'].apply(lambda x: np.sqrt(x) if x >= 0 else 0)) ,
                            'boot_std_mean':        np.mean(results['boot_var'].apply(lambda x: np.sqrt(x) if x >= 0 else 0)) ,
               }

(0.25, 0.2)

In [34]:
results['ijk_wager_var'][2] = -2

results['ijk_wager_var'].apply(lambda x: np.sqrt(x) if x >= 0 else 0)

0      0.0
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
995    0.0
996    0.0
997    0.0
998    0.0
999    0.0
Name: ijk_wager_var, Length: 1000, dtype: float64

In [ ]:


    non_neg_var_ijk_est = ijk_var_pred_X_point.copy()
    non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
    # erst die wurzel ziehen und dann den mittelwert bilden
    a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
    f.write(f'IJK STD (for RF) Mean-est               : {a:.4f}  \n rel. Abweichung zu emp. std {(a-b)/b*100:.4f} % \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.4f}\n\n')
    
    non_neg_var_ijk_est = ijk_var_biased_X_point.copy()
    non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
    # erst die wurzel ziehen und dann den mittelwert bilden
    a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
    f.write(f'IJK STD - biased (for RF) Mean-est               : {a:.4f}  \n rel. Abweichung zu emp. std {(a-b)/b*100:.4f} % \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.4f}\n\n')
    
    non_neg_var_ijk_est = jk_ab_var_pred_X_point.copy()
    non_neg_var_ijk_est[non_neg_var_ijk_est<0] = 0
    # erst die wurzel ziehen und dann den mittelwert bilden
    a = np.mean(np.sqrt(  non_neg_var_ijk_est    ))
    f.write(f'JK-AB(un-weighted) STD (for RF) Mean-est: {a:.4f} \n rel. Abweichung zu emp. std {(a-b)/b*100:.4f} %  \n std. des schätzers {np.std(np.sqrt(  non_neg_var_ijk_est    )):.4f} \n\n')

    a = np.mean(np.sqrt(  bootstrap_var_pred_X_point    ))
    f.write(f'Boot STD (for RF) Mean-est              : {a:.4f} \n rel. Abweichung zu emp. std {(a-b)/b*100:.4f} %  \n std. des schätzers {np.std(np.sqrt(  a    )):.4f} \n\n')




    f.write('\n\n### mean Data Stats over all simulations: ###\n')
    f.write('Number of simulations: '+str(n_sim)+'\n')
    f.write('cut-off time tau: '+str(tau)+'\n\n')
    f.write(f'Train ({int(n*0.7)}):\n')
    f.write(f'Events:     {round(np.mean(portion_events_after_cut_train)*100,2)}  %,  n={round(n*0.7*np.mean(portion_events_after_cut_train),0)}\n')
    f.write(f'No Events:  {round(np.mean(portion_no_events_after_cut_train)*100,2)} %,  n={round(n*0.7*np.mean(portion_no_events_after_cut_train),0)}\n')
    f.write(f'Censored:   {round(np.mean(portion_censored_after_cut_train)*100,2)} %,  n={round(n*0.7*np.mean(portion_censored_after_cut_train),0)}\n')
    f.write(f'Test  ({int(n*0.3)}):\n')
    f.write(f'Events:     {round(np.mean(portion_events_after_cut_test)*100,2)}  %,   n={round(n*0.3*np.mean(portion_events_after_cut_test),0)}\n')
    f.write(f'No Events:  {round(np.mean(portion_no_events_after_cut_test)*100,2)} %,   n={round(n*0.3*np.mean(portion_no_events_after_cut_test),0)}\n')
    f.write(f'Censored:   {round(np.mean(portion_censored_after_cut_test)*100,2)}  %,   n={round(n*0.3*np.mean(portion_censored_after_cut_test),0)}\n')


    f.write('\n\n### Evaluation: ###\n')
    f.write(f'WB C-Index IPCW: {np.mean(wb_cindex_ipcw):.4f}\n')
    f.write(f'WB MSE IPCW: {np.mean(wb_mse_ipcw):.4f}\n')
    f.write(f'RF MSE IPCW: {np.mean(rf_mse_ipcw):.4f}\n')
    f.write('\n')

    f.write('\n###Prediction Results:###\n')
    S_t = calculate_true_survival_probability(X_erwartung, params_data_creation, tau)
    f.write(f'True Y: {S_t:.4}f\n')
    f.write(f'WB Y_pred: {np.mean(wb_y_pred_X_point):.4f}\n')
    f.write(f'RF Y_pred: {np.mean(rf_y_pred_X_point):.4f}\n')
    f.write('\n')



    f.write('\n\n### Simulation Parameters: ###\n')
    f.write('first_level_B for bootstrap variance estimation (B_1): '+str(B_1)+'\n')

    f.write('Data Creation Parameters:\n')
    f.write(str(params_data_creation))
    
    f.write('\nRandom Forest Parameter:\n')
    f.write(str(params_rf)+'\n')
    f.write(f'the above seeds ({seed}) are start_seed for the simulation function\n')

print(experiment_name+'   finished')